# Hyperparameter Tuning Process

## 1. Core Idea and Train/Dev/Test Workflow

Hyperparameter tuning is the process of systematically choosing the external settings that control how a neural network learns. In neural networks, we distinguish between **parameters** and **hyperparameters**.

Parameters are learned directly by the model:

$$
W^{[l]}, b^{[l]}
$$

Hyperparameters are chosen by the practitioner:

$$
\alpha, \lambda, \beta, \beta_1, \beta_2, \epsilon, L, n^{[l]}, \text{mini-batch size}, \text{keep\_prob}
$$

Training optimizes the parameters:

$$
\theta^* = \arg\min_\theta J_{\text{train}}(\theta)
$$

where:

$$
\theta = \{W^{[1]}, b^{[1]}, \dots, W^{[L]}, b^{[L]}\}
$$

Hyperparameter tuning chooses the learning setup:

$$
h^* = \arg\min_h J_{\text{dev}}(\theta_h)
$$

where:

- $h$ is a hyperparameter configuration,
- $\theta_h$ is the set of learned parameters after training with $h$,
- $J_{\text{dev}}$ is the loss on the development set.

The essence is:

> Training chooses $W, b$.  
> Hyperparameter tuning chooses the conditions under which $W, b$ are learned well.

A correct deep learning workflow separates data into:

$$
\text{train set}, \quad \text{dev set}, \quad \text{test set}
$$

The train set is used to learn parameters. The dev set is used to choose hyperparameters. The test set is used only once at the end to estimate final generalization performance.

Correct workflow:

$$
\text{Train set} \rightarrow \text{learn parameters}
$$

$$
\text{Dev set} \rightarrow \text{select hyperparameters}
$$

$$
\text{Test set} \rightarrow \text{final evaluation}
$$

One should not tune hyperparameters on the test set. If the test set is repeatedly used during model selection, it becomes part of the development process, and the final test performance becomes overly optimistic.

## 2. Important Hyperparameters

The most important hyperparameter is usually the learning rate $\alpha$. The basic update rule is:

$$
W^{[l]} := W^{[l]} - \alpha dW^{[l]}
$$

$$
b^{[l]} := b^{[l]} - \alpha db^{[l]}
$$

If $\alpha$ is too small, training becomes very slow. If $\alpha$ is too large, the optimization process may oscillate or diverge. The learning rate controls the step size in the loss landscape, so it strongly affects whether the model can converge efficiently.

Another important hyperparameter is the L2 regularization parameter $\lambda$. For L2 regularization, the regularized cost is:

$$
J_{\text{reg}} = J + \frac{\lambda}{2m}\sum_l \|W^{[l]}\|_F^2
$$

The gradient becomes:

$$
dW^{[l]}_{\text{reg}} = dW^{[l]} + \frac{\lambda}{m}W^{[l]}
$$

The update rule is:

$$
W^{[l]} := W^{[l]} - \alpha \left(dW^{[l]} + \frac{\lambda}{m}W^{[l]}\right)
$$

This can be rewritten as:

$$
W^{[l]} := \left(1 - \frac{\alpha \lambda}{m}\right)W^{[l]} - \alpha dW^{[l]}
$$

This shows why L2 regularization is closely related to weight decay: it shrinks weights during training. If $\lambda$ is too small, the model may overfit. If $\lambda$ is too large, the model may underfit.

Mini-batch size controls how many examples are used for each gradient update. Using Andrew Ng's notation, for a mini-batch:

$$
X^{\{t\}} \in \mathbb{R}^{n_x \times m_b}
$$

$$
Y^{\{t\}} \in \mathbb{R}^{n_y \times m_b}
$$

where $m_b$ is the mini-batch size.

A small mini-batch gives noisy gradients but more frequent updates. A large mini-batch gives more stable gradients but fewer updates per epoch.

In PyTorch, the convention is usually different:

$$
X^{\{t\}} \in \mathbb{R}^{m_b \times n_x}
$$

This means each row is one training example, while in Andrew Ng's notation, each column is one training example.

For momentum:

$$
v_{dW} := \beta v_{dW} + (1-\beta)dW
$$

$$
v_{db} := \beta v_{db} + (1-\beta)db
$$

Then:

$$
W := W - \alpha v_{dW}
$$

$$
b := b - \alpha v_{db}
$$

The hyperparameter $\beta$ controls how much past gradients influence the current update. A common value is:

$$
\beta = 0.9
$$

This roughly averages over:

$$
\frac{1}{1-\beta} = \frac{1}{1-0.9} = 10
$$

recent gradients.

Adam combines momentum and RMSprop:

$$
v_{dW} := \beta_1 v_{dW} + (1-\beta_1)dW
$$

$$
s_{dW} := \beta_2 s_{dW} + (1-\beta_2)(dW)^2
$$

Bias correction:

$$
v_{dW}^{\text{corrected}} = \frac{v_{dW}}{1-\beta_1^t}
$$

$$
s_{dW}^{\text{corrected}} = \frac{s_{dW}}{1-\beta_2^t}
$$

Update:

$$
W := W - \alpha \frac{v_{dW}^{\text{corrected}}}{\sqrt{s_{dW}^{\text{corrected}}}+\epsilon}
$$

Common default values are:

$$
\beta_1 = 0.9, \quad \beta_2 = 0.999, \quad \epsilon = 10^{-8}
$$

In practice, when using Adam, the learning rate $\alpha$ is usually tuned first, while $\beta_1$, $\beta_2$, and $\epsilon$ are often kept at their default values initially.

## 3. Search Strategy: Random Search, Coarse-to-Fine Search, and Scale

Grid search tries fixed combinations of hyperparameters. For example:

$$
\alpha \in \{0.0001, 0.001, 0.01\}
$$

$$
\lambda \in \{0, 0.1, 1\}
$$

This creates a fixed grid of experiments. However, in deep learning, random search is often more efficient than grid search because not all hyperparameters are equally important.

For example, if learning rate is very important and another hyperparameter is not very sensitive, grid search wastes many trials on the less important dimension. Random search samples more diverse values for each hyperparameter and is more likely to find good values for the important ones.

The practical idea is:

> Do not spend all trials evenly across every dimension.  
> Use random search to explore more values of the sensitive hyperparameters.

A good tuning process often happens in two stages. First, perform a coarse search over a wide range. For example:

$$
\alpha \in [10^{-5}, 10^{-1}]
$$

After observing which region performs well, zoom in. If good learning rates are found around:

$$
[10^{-3}, 10^{-2}]
$$

then perform a finer search in that smaller region:

$$
\alpha \in [10^{-3}, 10^{-2}]
$$

This is called coarse-to-fine search. The goal is not to find the perfect value immediately. The goal is first to identify a promising region, then search more carefully inside it.

Different hyperparameters should be sampled using different scales. The learning rate should usually be sampled on a logarithmic scale. Instead of sampling linearly:

$$
0.1, 0.2, 0.3, 0.4
$$

we usually sample values such as:

$$
10^{-4}, 10^{-3}, 10^{-2}, 10^{-1}
$$

A good way to sample is:

$$
r \sim U(-4, -1)
$$

$$
\alpha = 10^r
$$

Example:

~~~python
import numpy as np

learning_rate = 10 ** np.random.uniform(-4, -1)
print(learning_rate)
~~~

This samples learning rates from:

$$
[10^{-4}, 10^{-1}]
$$

on a logarithmic scale.

The L2 regularization parameter $\lambda$ should also often be sampled on a logarithmic scale:

$$
\lambda \in [10^{-5}, 10^1]
$$

Sample:

$$
r \sim U(-5, 1)
$$

$$
\lambda = 10^r
$$

Example:

~~~python
import numpy as np

lambd = 10 ** np.random.uniform(-5, 1)
print(lambd)
~~~

For exponentially weighted averages, values such as:

$$
\beta = 0.9, 0.99, 0.999
$$

look close numerically, but they behave very differently:

$$
\frac{1}{1-0.9} = 10
$$

$$
\frac{1}{1-0.99} = 100
$$

$$
\frac{1}{1-0.999} = 1000
$$

Therefore, if tuning $\beta$, it is often better to sample $1-\beta$ on a logarithmic scale:

$$
1-\beta \in [10^{-3}, 10^{-1}]
$$

Sample:

$$
r \sim U(-3, -1)
$$

$$
\beta = 1 - 10^r
$$

Example:

~~~python
import numpy as np

r = np.random.uniform(-3, -1)
beta = 1 - 10 ** r
print(beta)
~~~

## 4. Practical Workflow and Diagnosis

A practical tuning workflow is:

```text
1. Verify that the implementation is correct.
2. Check whether the model can overfit a small subset.
3. Tune the learning rate first.
4. Use bias/variance analysis to decide the next direction.
5. Tune regularization if the model overfits.
6. Tune architecture size if the model underfits.
7. Fine-tune batch size, optimizer, and learning rate schedule.
8. Select the final model using the dev set.
9. Evaluate once on the test set.
```

Before tuning, it is important to make sure the implementation is correct. A useful debugging test is:

```text
Try to overfit 50–100 training examples.
```

If the model cannot overfit a very small training set, then there may be a bug in forward propagation, backward propagation, loss computation, optimizer update, activation function, label encoding, or data preprocessing. Tuning is only meaningful after the training pipeline is reasonably correct.

Hyperparameter tuning should also be guided by bias and variance analysis. If both train accuracy and dev accuracy are low:

```text
Train accuracy: low
Dev accuracy: low
```

the model may have high bias or optimization problems. Possible solutions include increasing model size, training longer, reducing regularization, improving initialization, using Adam or momentum, or using Batch Normalization.

If train accuracy is high but dev accuracy is low:

```text
Train accuracy: high
Dev accuracy: low
```

the model has high variance. Possible solutions include increasing L2 regularization, using dropout, using data augmentation, reducing model size, using early stopping, or collecting more data.

If both train and dev performance are good:

```text
Train accuracy: high
Dev accuracy: high
```

the model is likely in a good region, and further tuning can be done more carefully.

## 5. Implementation

### A. PyTorch

Below is a simple PyTorch-style tuning setup.

~~~python
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

def sample_hyperparameters():
    config = {
        "learning_rate": 10 ** np.random.uniform(-5, -1),
        "weight_decay": 10 ** np.random.uniform(-6, -1),
        "hidden_units": random.choice([64, 128, 256, 512]),
        "batch_size": random.choice([32, 64, 128, 256]),
        "num_layers": random.choice([2, 3, 4]),
    }
    return config
~~~

In PyTorch, input data usually has shape:

$$
X \in \mathbb{R}^{m \times n_x}
$$

where $m$ is the number of examples. This differs from Andrew Ng's convention:

$$
X \in \mathbb{R}^{n_x \times m}
$$

A simple MLP can be defined as:

~~~python
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_units, output_dim, num_layers):
        super().__init__()

        layers = []
        layers.append(nn.Linear(input_dim, hidden_units))
        layers.append(nn.ReLU())

        for _ in range(num_layers - 1):
            layers.append(nn.Linear(hidden_units, hidden_units))
            layers.append(nn.ReLU())

        layers.append(nn.Linear(hidden_units, output_dim))

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)
~~~

A training function may look like:

~~~python
def train_one_config(train_dataset, dev_loader, input_dim, output_dim, config, epochs=10):
    train_loader = DataLoader(
        train_dataset,
        batch_size=config["batch_size"],
        shuffle=True
    )

    model = MLP(
        input_dim=input_dim,
        hidden_units=config["hidden_units"],
        output_dim=output_dim,
        num_layers=config["num_layers"]
    )

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"]
    )

    for epoch in range(epochs):
        model.train()

        for X_batch, y_batch in train_loader:
            logits = model(X_batch)
            loss = criterion(logits, y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    dev_acc = evaluate_accuracy(model, dev_loader)
    return dev_acc
~~~

The tuning loop:

~~~python
results = []

for trial in range(30):
    config = sample_hyperparameters()

    dev_acc = train_one_config(
        train_dataset=train_dataset,
        dev_loader=dev_loader,
        input_dim=input_dim,
        output_dim=output_dim,
        config=config,
        epochs=10
    )

    results.append({
        "config": config,
        "dev_acc": dev_acc
    })

    print(f"Trial {trial + 1}")
    print("Config:", config)
    print("Dev accuracy:", dev_acc)
    print()
~~~

Selecting the best result:

~~~python
best_result = max(results, key=lambda x: x["dev_acc"])

print("Best config:")
print(best_result["config"])
print("Best dev accuracy:")
print(best_result["dev_acc"])
~~~

### B. TensorFlow

Below is a simple TensorFlow/Keras version.

~~~python
import random
import numpy as np
import tensorflow as tf

def sample_hyperparameters():
    config = {
        "learning_rate": 10 ** np.random.uniform(-5, -1),
        "weight_decay": 10 ** np.random.uniform(-6, -1),
        "hidden_units": random.choice([64, 128, 256, 512]),
        "batch_size": random.choice([32, 64, 128, 256]),
        "num_layers": random.choice([2, 3, 4]),
    }
    return config
~~~

A simple MLP model:

~~~python
def build_model(input_dim, output_dim, config):
    model = tf.keras.Sequential()

    model.add(tf.keras.layers.Input(shape=(input_dim,)))

    for _ in range(config["num_layers"]):
        model.add(tf.keras.layers.Dense(
            config["hidden_units"],
            activation="relu",
            kernel_regularizer=tf.keras.regularizers.l2(config["weight_decay"])
        ))

    model.add(tf.keras.layers.Dense(output_dim, activation="softmax"))

    optimizer = tf.keras.optimizers.Adam(
        learning_rate=config["learning_rate"]
    )

    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model
~~~

Training one configuration:

~~~python
def train_one_config(X_train, y_train, X_dev, y_dev, input_dim, output_dim, config):
    model = build_model(input_dim, output_dim, config)

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_dev, y_dev),
        batch_size=config["batch_size"],
        epochs=10,
        verbose=0
    )

    dev_acc = history.history["val_accuracy"][-1]
    return dev_acc
~~~

Tuning loop:

~~~python
results = []

for trial in range(30):
    config = sample_hyperparameters()

    dev_acc = train_one_config(
        X_train=X_train,
        y_train=y_train,
        X_dev=X_dev,
        y_dev=y_dev,
        input_dim=input_dim,
        output_dim=output_dim,
        config=config
    )

    results.append({
        "config": config,
        "dev_acc": dev_acc
    })

    print(f"Trial {trial + 1}")
    print("Config:", config)
    print("Dev accuracy:", dev_acc)
    print()
~~~

Selecting the best configuration:

~~~python
best_result = max(results, key=lambda x: x["dev_acc"])

print("Best config:")
print(best_result["config"])
print("Best dev accuracy:")
print(best_result["dev_acc"])
~~~

Important parameters:

- `learning_rate`: controls the step size of parameter updates.
- `weight_decay`: plays a role similar to L2 regularization.
- `hidden_units`: controls the width of hidden layers.
- `num_layers`: controls the depth of the network.
- `batch_size`: controls the number of examples used in each gradient update.
- `epochs`: controls how many times the model sees the training set.

## 6. Common Mistakes and Final Takeaway

A common mistake is tuning before debugging. If the implementation is wrong, hyperparameter tuning is meaningless. Before tuning, verify that the loss decreases, gradients are reasonable, the model can overfit a small subset, labels are encoded correctly, and preprocessing is correct.

Another serious mistake is tuning on the test set. Correct usage is:

```text
Train set: learn parameters
Dev set: tune hyperparameters
Test set: final evaluation
```

Using the test set repeatedly causes data leakage.

Another common mistake is sampling learning rate on a linear scale. A bad search space is:

$$
\alpha \in \{0.1, 0.2, 0.3, 0.4\}
$$

A better search space is:

$$
\alpha \in \{10^{-4}, 10^{-3}, 10^{-2}, 10^{-1}\}
$$

Learning rate should usually be searched on a logarithmic scale.

It is also important not to look only at accuracy. You should monitor train loss, dev loss, train accuracy, and dev accuracy. If train loss decreases but dev loss increases, the model is likely overfitting. If train loss does not decrease, the issue may be learning rate, model capacity, or implementation.

Finally, do not confuse optimization problems with generalization problems. If train loss is high, the model may have high bias or poor optimization. If train loss is low but dev loss is high, the model is overfitting.

Hyperparameter tuning connects directly to earlier topics from Course 2. Bias and variance tell us what to tune. L2 regularization and dropout control overfitting. Initialization affects gradient flow. Mini-batch gradient descent, momentum, RMSprop, Adam, and learning rate decay improve optimization.

The most important practical rule is:

$$
\boxed{\text{Tune learning rate first.}}
$$

Then use bias and variance analysis to decide what to tune next.

If the model underfits, consider:

$$
\text{larger model}, \quad \text{less regularization}, \quad \text{better optimization}
$$

If the model overfits, consider:

$$
\text{L2 regularization}, \quad \text{dropout}, \quad \text{data augmentation}, \quad \text{smaller model}
$$

The essence is:

> Training optimizes parameters.  
> Hyperparameter tuning optimizes the learning process itself.

A strong deep learning engineer does not merely train one model. A strong deep learning engineer designs, tests, compares, and improves the full training system using disciplined hyperparameter tuning.

# Using an Appropriate Scale to Pick Hyperparameters

## 1. Core Idea

When tuning hyperparameters, choosing a good range is not enough. We also need to choose the appropriate **scale** for sampling values.

Some hyperparameters should be sampled on a linear scale, while others should be sampled on a logarithmic scale. The main reason is that many important hyperparameters in deep learning affect training by **orders of magnitude**, not by equal linear differences.

For example, the difference between $10^{-4}$ and $10^{-3}$ is significant because it is a $10\times$ increase. Similarly, the difference between $10^{-3}$ and $10^{-2}$ is also a $10\times$ increase.

Therefore, for hyperparameters such as learning rate and regularization strength, we usually care more about multiplicative changes than additive changes.

The key idea is:

$$
\boxed{\text{Good hyperparameter tuning requires a good scale, not just a good range.}}
$$

## 2. Learning Rate and Log Scale

The learning rate $\alpha$ is usually the most important hyperparameter to tune.

The parameter update rule is:

$$
W^{[l]} := W^{[l]} - \alpha dW^{[l]}
$$

$$
b^{[l]} := b^{[l]} - \alpha db^{[l]}
$$

If $\alpha$ is too small, training becomes very slow. If $\alpha$ is too large, the optimization process may oscillate or diverge.

For learning rate, we usually do not sample linearly from a range such as:

$$
[0.0001, 1]
$$

because most sampled values would be too large for many deep learning problems.

Instead, we sample on a logarithmic scale. For example, to search:

$$
\alpha \in [10^{-4}, 10^0]
$$

we sample:

$$
r \sim U(-4, 0)
$$

and set:

$$
\alpha = 10^r
$$

In Python:

~~~python
import numpy as np

alpha = 10 ** np.random.uniform(-4, 0)
print(alpha)
~~~

This gives a learning rate sampled across several orders of magnitude.

Common learning rate candidates are often around:

$$
10^{-4}, 10^{-3}, 10^{-2}, 10^{-1}
$$

The practical reason is that learning rate is sensitive to multiplicative changes. A learning rate of $10^{-3}$ can behave very differently from $10^{-2}$, even though the absolute difference may look small.

## 3. Regularization Parameter and Log Scale

The L2 regularization parameter $\lambda$ should also often be sampled on a logarithmic scale.

With L2 regularization, the regularized cost is:

$$
J_{\text{reg}} = J + \frac{\lambda}{2m}\sum_l \|W^{[l]}\|_F^2
$$

The gradient becomes:

$$
dW^{[l]}_{\text{reg}} = dW^{[l]} + \frac{\lambda}{m}W^{[l]}
$$

The update rule is:

$$
W^{[l]} := W^{[l]} - \alpha \left(dW^{[l]} + \frac{\lambda}{m}W^{[l]}\right)
$$

If $\lambda$ is too small, regularization may be too weak and the model may overfit. If $\lambda$ is too large, regularization may be too strong and the model may underfit.

A reasonable search range may be:

$$
\lambda \in [10^{-5}, 10^1]
$$

To sample from this range, we use:

$$
r \sim U(-5, 1)
$$

$$
\lambda = 10^r
$$

In Python:

~~~python
import numpy as np

lambd = 10 ** np.random.uniform(-5, 1)
print(lambd)
~~~

This allows us to explore very small, medium, and large regularization strengths in a balanced way.

## 4. Scale for Momentum Parameter Beta

The momentum parameter $\beta$ requires a slightly different treatment.

For exponentially weighted averages, we have:

$$
v_t = \beta v_{t-1} + (1-\beta)\theta_t
$$

The value of $\beta$ is usually close to $1$, such as:

$$
0.9,\quad 0.99,\quad 0.999
$$

Although these values look close numerically, they behave very differently.

A useful approximation is:

$$
\frac{1}{1-\beta}
$$

This tells us roughly how many recent values the exponentially weighted average remembers.

For example:

$$
\beta = 0.9
$$

corresponds roughly to averaging over:

$$
\frac{1}{1-0.9} = 10
$$

recent values.

For $\beta = 0.99$:

$$
\frac{1}{1-0.99} = 100
$$

For $\beta = 0.999$:

$$
\frac{1}{1-0.999} = 1000
$$

Therefore, instead of sampling $\beta$ directly, it is better to sample $1-\beta$ on a logarithmic scale.

For example:

$$
1-\beta \in [10^{-3}, 10^{-1}]
$$

Sample:

$$
r \sim U(-3, -1)
$$

Then:

$$
\beta = 1 - 10^r
$$

In Python:

~~~python
import numpy as np

beta = 1 - 10 ** np.random.uniform(-3, -1)
print(beta)
~~~

This gives a better coverage of values such as $0.9$, $0.99$, and $0.999$.

## 5. Hyperparameters That Are Usually Sampled Discretely

Not every hyperparameter should be sampled on a logarithmic scale.

Some hyperparameters are naturally discrete or categorical.

For example, mini-batch size is usually selected from a fixed set:

$$
\text{mini-batch size} \in \{16, 32, 64, 128, 256, 512\}
$$

In Python:

~~~python
import random

batch_size = random.choice([16, 32, 64, 128, 256, 512])
~~~

The number of hidden units can also be selected from a discrete set:

$$
n^{[l]} \in \{64, 128, 256, 512, 1024\}
$$

In Python:

~~~python
hidden_units = random.choice([64, 128, 256, 512, 1024])
~~~

The number of layers is also discrete:

$$
L \in \{2, 3, 4, 5\}
$$

In Python:

~~~python
num_layers = random.choice([2, 3, 4, 5])
~~~

For these hyperparameters, categorical sampling is usually more appropriate than continuous sampling.

## 6. Practical Search Space Example

A good random search setup combines different sampling strategies for different hyperparameters.

~~~python
import random
import numpy as np

config = {
    "learning_rate": 10 ** np.random.uniform(-4, -1),
    "lambd": 10 ** np.random.uniform(-5, 1),
    "beta": 1 - 10 ** np.random.uniform(-3, -1),
    "batch_size": random.choice([32, 64, 128, 256]),
    "hidden_units": random.choice([64, 128, 256, 512]),
}
~~~

In this example:

- `learning_rate` is sampled on a log scale.
- `lambd` is sampled on a log scale.
- `beta` is sampled by applying log scale to $1-\beta$.
- `batch_size` is sampled from a discrete set.
- `hidden_units` is sampled from a discrete set.

This reflects the main lesson:

> Different hyperparameters require different sampling scales.

## 7. Essential Conclusion

Using an appropriate scale is a key part of hyperparameter tuning.

For learning rate:

$$
\alpha = 10^r,\quad r \sim U(a,b)
$$

For L2 regularization:

$$
\lambda = 10^r,\quad r \sim U(a,b)
$$

For momentum:

$$
\beta = 1 - 10^r,\quad r \sim U(a,b)
$$

The main reason is that these hyperparameters often affect training by multiplicative factors, not additive differences.

The core takeaway is:

$$
\boxed{\text{Do not sample hyperparameters only by their raw numerical range.}}
$$

Instead, sample them using the scale that matches how they affect learning.

In practice:

- use log scale for learning rate,
- use log scale for L2 regularization,
- use log scale on $1-\beta$ for momentum,
- use categorical choices for mini-batch size, hidden units, and number of layers.

A good search space is not only about where to search. It is also about how to sample.

# Hyperparameters Tuning in Practice: Pandas vs. Caviar

## 1. Core Idea

In practical deep learning, hyperparameter tuning is not only about choosing values such as learning rate, regularization strength, batch size, or model size. It is also about deciding **how to organize experiments**.

Andrew Ng uses two metaphors:

$$
\text{Panda approach}
$$

and

$$
\text{Caviar approach}
$$

These two approaches describe different ways to allocate computational resources during hyperparameter tuning.

The main question is:

> Should we train a small number of models carefully, or should we train many models in parallel and choose the best one?

## 2. Panda Approach

The Panda approach means training **one or only a few models at a time**, while carefully monitoring their learning behavior.

It is called the Panda approach because pandas have very few babies and take care of them carefully. Similarly, in this tuning strategy, we “babysit” one model carefully.

A typical Panda-style workflow is:

```text
Train one model
Monitor train loss, dev loss, train accuracy, and dev accuracy
Analyze the learning curve
Adjust hyperparameters based on the observed behavior
Continue training or stop early if the trial is clearly bad
```

For example, suppose we train a model with:

$$
\alpha = 10^{-3}, \quad \lambda = 10^{-4}, \quad \text{batch size} = 64
$$

If the training loss decreases very slowly, the learning rate may be too small. We may stop the trial early and try a larger learning rate.

If the loss explodes or becomes NaN, the learning rate may be too large. We should stop the trial and reduce the learning rate.

If the training accuracy becomes high but the dev accuracy remains low, the model is probably overfitting. We may increase regularization, add dropout, use data augmentation, or apply early stopping.

The Panda approach is especially useful when:

- each training run is expensive,
- the model is large,
- compute resources are limited,
- the training pipeline is still being debugged,
- careful learning-curve analysis is needed.

The important point is:

> Panda approach does not mean waiting until the full training run finishes.  
> If the learning curve clearly shows that a trial is bad, we can stop early and adjust the hyperparameters.

## 3. Caviar Approach

The Caviar approach means training **many models with different hyperparameter configurations**, often in parallel, and then selecting the best configuration based on dev set performance.

It is called the Caviar approach because caviar contains many eggs. Similarly, this strategy produces many training trials.

A typical Caviar-style workflow is:

```text
Define a search space
Sample many hyperparameter configurations
Train many models, often in parallel
Evaluate each model on the dev set
Select the best configurations
Search more carefully around promising regions
```

For one configuration $h$, training produces learned parameters $\theta_h$. The goal is to choose the configuration that performs best on the dev set:

$$
h^* = \arg\min_h J_{\text{dev}}(\theta_h)
$$

or, if using accuracy:

$$
h^* = \arg\max_h \text{Accuracy}_{\text{dev}}(\theta_h)
$$

The Caviar approach is useful when:

- each model trains quickly,
- many experiments can run in parallel,
- enough compute resources are available,
- the training pipeline is already reliable,
- we want to explore a broad search space.

This approach works naturally with random search, coarse-to-fine search, and tools such as Optuna.

## 4. Practical Strategy

In real projects, the best strategy is often not purely Panda or purely Caviar. A strong practical workflow is:

```text
Panda → Caviar → Panda
```

At the beginning, use the Panda approach to debug the training pipeline. The goal is to verify that the model can learn, the loss decreases properly, the data preprocessing is correct, and the labels are encoded correctly.

After the pipeline is stable, use the Caviar approach to explore many hyperparameter configurations. This can be done with random search, Optuna, or another tuning method.

After finding several promising configurations, return to the Panda approach. Carefully inspect the top models, analyze their learning curves, check for overfitting or instability, and fine-tune around the best regions.

A good practical workflow is:

```text
1. Build a baseline model.
2. Use Panda-style tuning to debug the training pipeline.
3. Tune the learning rate roughly.
4. Use Caviar-style tuning to explore the search space.
5. Select top configurations using the dev set.
6. Inspect top models carefully.
7. Fine-tune around promising regions.
8. Evaluate the final model once on the test set.
```

## 5. Connection to Bias, Variance, and Search Strategy

The Panda approach is especially useful for diagnosing bias and variance.

If both train and dev performance are poor, the model may have high bias or optimization problems. Possible responses include increasing model capacity, training longer, reducing regularization, improving initialization, using Adam or momentum, or applying Batch Normalization.

If train performance is good but dev performance is poor, the model has high variance. Possible responses include increasing L2 regularization, adding dropout, using data augmentation, early stopping, or reducing model size.

The Caviar approach is useful for broad exploration. It helps identify patterns across many trials, such as:

```text
Learning rates above a certain value often diverge.
Very small regularization values often overfit.
A certain batch size works better than others.
A certain model size gives the best dev performance.
```

Caviar works best when the search space is designed properly. For example:

- learning rate should usually be sampled on a log scale,
- L2 regularization should usually be sampled on a log scale,
- $1-\beta$ should be sampled on a log scale when tuning momentum,
- batch size and hidden units are often sampled from categorical choices.

## 6. Essential Conclusion

The main lesson of **Pandas vs. Caviar** is that hyperparameter tuning depends on computational resources and experimental strategy.

If training is expensive or the pipeline is not stable, use the Panda approach:

$$
\text{Panda} = \text{careful sequential tuning}
$$

If training is cheap or many trials can be run in parallel, use the Caviar approach:

$$
\text{Caviar} = \text{broad parallel tuning}
$$

The best practical strategy is often:

$$
\boxed{\text{Panda} \rightarrow \text{Caviar} \rightarrow \text{Panda}}
$$

Use Panda to debug, Caviar to explore, and Panda again to fine-tune and analyze the best models.

The essence is:

> Hyperparameter tuning is not only about choosing values.  
> It is about organizing experiments intelligently based on compute, training cost, and model behavior.